# CustomerPulse: SaaS Customer Churn Intelligence Platform

## Project Overview
Customer retention is one of the most important business problems faced by subscription based companies. Acquiring a new customer is often significantly more expensive than retaining an existing one. Therefore, companies need systems that can identify customers who are likely to churn and take preventive action.

## Aim of the Project
The aim of this project is to build an end to end customer churn prediction pipeline and convert the results into actionable business intelligence through the CustomerPulse dashboard.

## Objectives
1. Clean and preprocess the customer dataset.
2. Engineer meaningful business features.
3. Train and evaluate multiple machine learning models.
4. Identify the strongest predictors of churn.
5. Generate customer risk scores and health metrics.
6. Produce a dataset that can be consumed by a dashboard application.

## Technologies Used
- Python
- Pandas
- NumPy
- Scikit-learn
- Matplotlib
- Seaborn
- Jupyter Notebook

## Project Workflow
Dataset Collection → Data Cleaning → Feature Engineering → Model Training → Model Evaluation → Prediction Generation → Dashboard Integration


In [3]:
# pip install -r requirements.txt

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)

pd.set_option('display.max_columns', None)


# Chapter 1: Dataset Collection and Understanding

## Objective
In this step, the customer churn dataset is loaded into the notebook so that it can be explored and prepared for machine learning.

## Why this step is important
Machine learning models can only work after the data has been imported correctly. Before building any predictive system, we must first understand:
- Number of records available
- Number of features available
- Data types of each column
- Initial quality of the dataset

## Expected Outcome
After executing this cell, the dataset will be stored inside a Pandas DataFrame called `df`. The first few rows are displayed to verify that the data has been loaded correctly.


In [5]:
df = pd.read_csv('../input_data/Telco_customer_churn.csv')

print(df.shape)
df.head()


(7043, 33)


,CustomerID,Count,Country,State,City,ZipCode,LatLong,Latitude,Longitude,Gender,SeniorCitizen,Partner,Dependents,TenureMonths,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,ChurnLabel,ChurnValue,ChurnScore,CLTV,ChurnReason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


# Chapter 2: Data Cleaning and Preprocessing

## Objective
Real world datasets usually contain missing values, incorrect data types and inconsistent formatting. This step prepares the data for modelling.

## Tasks Performed
1. Standardize column names.
2. Convert numerical columns into proper numeric data types.
3. Remove or handle missing values.
4. Prepare the dataset for feature engineering.

## Why this step is important
Machine learning algorithms are highly sensitive to poor quality data. Incorrect data cleaning can significantly reduce model performance and lead to misleading predictions.


In [6]:
df.columns = df.columns.str.strip()

# Convert TotalCharges safely
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(
        df['TotalCharges'],
        errors='coerce'
    )
    df['TotalCharges'] = df['TotalCharges'].fillna(0)

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CustomerID        7043 non-null   object 
 1   Count             7043 non-null   int64  
 2   Country           7043 non-null   object 
 3   State             7043 non-null   object 
 4   City              7043 non-null   object 
 5   ZipCode           7043 non-null   int64  
 6   LatLong           7043 non-null   object 
 7   Latitude          7043 non-null   float64
 8   Longitude         7043 non-null   float64
 9   Gender            7043 non-null   object 
 10  SeniorCitizen     7043 non-null   object 
 11  Partner           7043 non-null   object 
 12  Dependents        7043 non-null   object 
 13  TenureMonths      7043 non-null   int64  
 14  PhoneService      7043 non-null   object 
 15  MultipleLines     7043 non-null   object 
 16  InternetService   7043 non-null   object 


# Chapter 3: Defining the Target Variable

## Objective
The purpose of this project is to predict whether a customer will churn. Therefore, we need a target variable called `y`.

## Explanation
- `1` represents a customer likely to churn.
- `0` represents a customer who stays with the company.

The target variable acts as the answer key that the machine learning model will learn from during training.


In [7]:
# Use ChurnValue if present, otherwise derive from ChurnLabel
if 'ChurnValue' in df.columns:
    y = df['ChurnValue']
else:
    y = (df['ChurnLabel'] == 'Yes').astype(int)

print(y.value_counts(normalize=True))


ChurnValue
0    0.73463
1    0.26537
Name: proportion, dtype: float64


# Chapter 4: Removing Data Leakage

## Objective
Some columns contain information that directly reveals the answer or contains identifiers that do not help prediction.

## What is Data Leakage?
Data leakage happens when the model accidentally gets information that would not be available in a real prediction scenario.

Examples:
- Customer IDs
- Geographic coordinates
- Columns generated after churn has already happened

Removing these columns helps the model learn genuine customer behaviour patterns.


In [8]:
drop_cols = [
    'Count',
    'Country',
    'State',
    'City',
    'ZipCode',
    'LatLong',
    'Latitude',
    'Longitude',
    'CustomerStatus',
    'ChurnLabel',
    'ChurnValue',
    'ChurnScore',
    'ChurnReason',
    'CLTV',
    'Quarter',
    'ReferredAFriend',
    'NumberOfReferrals',
    'Offer',
    'AvgMonthlyLongDistanceCharges',
    'AvgMonthlyGBDownload',
    'TotalRefunds',
    'TotalExtraDataCharges',
    'TotalLongDistanceCharges',
    'TotalRevenue'
]

X = df.drop(columns=drop_cols, errors='ignore')

customer_ids = None
if 'CustomerID' in X.columns:
    customer_ids = X['CustomerID']
    X = X.drop(columns=['CustomerID'])

print(X.shape)
X.head()


(7043, 19)


,Gender,SeniorCitizen,Partner,Dependents,TenureMonths,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30


# Chapter 5: Feature Engineering

## Objective
Feature engineering creates new variables from existing data that may provide stronger predictive power.

## Examples
- Average monthly spending ratio
- Contract based indicators
- Customer tenure related information

## Why it matters
Good features often improve model performance more than changing algorithms. This stage converts raw business information into meaningful predictors.


In [9]:
if {'TotalCharges', 'TenureMonths'}.issubset(df.columns):
    X['avg_monthly_spend_ratio'] = (
        df['TotalCharges'] / (df['TenureMonths'] + 1)
    )

if 'TenureMonths' in df.columns:
    X['is_new_customer'] = (
        df['TenureMonths'] <= 12
    ).astype(int)

if 'MonthlyCharges' in df.columns:
    X['high_monthly_spend'] = (
        df['MonthlyCharges'] >
        df['MonthlyCharges'].median()
    ).astype(int)

X.head()


,Gender,SeniorCitizen,Partner,Dependents,TenureMonths,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,avg_monthly_spend_ratio,is_new_customer,high_monthly_spend
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,36.050000,1,0
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,50.550000,1,1
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,91.166667,1,1
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,105.036207,0,1
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,100.726000,0,1


# Chapter 6: Encoding Categorical Variables

## Objective
Machine learning algorithms work with numbers and cannot directly understand text values such as:

- Male / Female
- Yes / No
- Month-to-month / Two year contract

Therefore, categorical columns are converted into numerical representations using one-hot encoding.


In [10]:
X = pd.get_dummies(
    X,
    drop_first=True
)

print(X.shape)


(7043, 33)


# Chapter 7: Splitting the Dataset

## Objective
The dataset is divided into:
- Training data
- Testing data

## Why do we do this?
The model should be evaluated on data it has never seen before. This gives a realistic estimate of how well the model will perform in production.

A common split is:
- 80% Training
- 20% Testing


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training:', X_train.shape)
print('Testing:', X_test.shape)


Training: (5634, 33)
Testing: (1409, 33)


# Chapter 8: Feature Scaling

## Objective
Some algorithms such as Logistic Regression perform better when numerical variables are placed on a similar scale.

## Why Scaling is Required
Without scaling, variables with large numerical values can dominate the learning process and reduce model performance.


In [12]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Chapter 9: Model Training

## Objective
Multiple machine learning models are trained and compared.

## Models Used
- Logistic Regression
- Random Forest
- Additional algorithms included in the notebook

## Project Goal
Instead of assuming one model is best, several models are trained and their performance is measured objectively.


In [13]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight='balanced'
    )
}

results = {}

for name, model in models.items():

    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)

    auc = roc_auc_score(y_test, y_pred_proba)

    results[name] = {
        'model': model,
        'auc': auc,
        'pred': y_pred,
        'proba': y_pred_proba
    }

    print(f'\n{name}')
    print(f'AUC: {auc:.4f}')
    print(classification_report(y_test, y_pred))



Logistic Regression
AUC: 0.8522
              precision    recall  f1-score   support

           0       0.90      0.74      0.81      1035
           1       0.52      0.78      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409


Random Forest
AUC: 0.8371
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.62      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409



# Chapter 10: Model Evaluation

## Objective
The trained models are compared using performance metrics.

## Metrics Used
- Accuracy
- Precision
- Recall
- F1 Score
- ROC AUC

The model with the strongest overall performance will later be selected for dashboard generation and customer risk prediction.


In [14]:
metrics_df = pd.DataFrame({
    'Model': list(results.keys()),
    'AUC': [v['auc'] for v in results.values()]
}).sort_values('AUC', ascending=False)

metrics_df


,Model,AUC
0,Logistic Regression,0.852195
1,Random Forest,0.837126


# Chapter 11: Feature Importance Analysis

## Objective
Understanding *why* customers churn is as important as predicting churn.

Feature importance analysis helps answer questions such as:
- Which factors influence churn the most?
- Which business variables should be monitored?
- Which retention strategies should be prioritised?

This information is later visualised inside the CustomerPulse dashboard.


In [15]:
rf = results['Random Forest']['model']

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(
    'Importance',
    ascending=False
)

feature_importance.head(20)


,Feature,Importance
2,TotalCharges,0.129289
0,TenureMonths,0.127176
3,avg_monthly_spend_ratio,0.117036
1,MonthlyCharges,0.115839
28,Contract_Two year,0.052564
9,Dependents_Yes,0.050349
13,InternetService_Fiber optic,0.040094
4,is_new_customer,0.034735
31,PaymentMethod_Electronic check,0.033718
27,Contract_One year,0.024321


# Chapter 12: Customer Risk Scoring and Dashboard Dataset Creation

## Objective
The best performing model is used to generate predictions for every customer.

## Outputs Produced
- Churn probability
- Risk segment
- Customer lifecycle stage
- Retention priority
- Customer health score

These outputs form the final analytical dataset that powers the CustomerPulse dashboard developed as part of this project.


In [16]:
best_model_name = metrics_df.iloc[0]['Model']

# Get churn probabilities from the best model
churn_probability = results[best_model_name]['proba']

# Build prediction table
prediction_table = pd.DataFrame({
    'CustomerID': customer_ids.loc[X_test.index].values,
    'ActualChurn': y_test.values,
    'ChurnProbability': churn_probability,
    'MonthlyCharges': df.loc[X_test.index, 'MonthlyCharges'].values,
    'TenureMonths': df.loc[X_test.index, 'TenureMonths'].values
})

# -----------------------------
# Risk Segmentation
# -----------------------------
prediction_table['RiskSegment'] = pd.cut(
    prediction_table['ChurnProbability'],
    bins=[0, 0.30, 0.60, 1.0],
    labels=[
        'Low Risk',
        'Moderate Risk',
        'High Risk'
    ]
)

# -----------------------------
# Customer Health Score
# Higher score = healthier customer
# -----------------------------
prediction_table['CustomerHealthScore'] = (
    (1 - prediction_table['ChurnProbability']) * 100
).round().astype(int)

# -----------------------------
# Retention Priority
# -----------------------------
prediction_table['RetentionPriority'] = np.select(
    [
        prediction_table['ChurnProbability'] >= 0.80,
        prediction_table['ChurnProbability'] >= 0.60,
        prediction_table['ChurnProbability'] >= 0.30
    ],
    [
        'Immediate Action',
        'Monitor Closely',
        'Nurture'
    ],
    default='Healthy'
)

# -----------------------------
# Estimated Annual Revenue at Risk
# Only for High-Risk Customers
# -----------------------------
prediction_table['RevenueAtRisk'] = np.where(
    prediction_table['RiskSegment'] == 'High Risk',
    prediction_table['MonthlyCharges'] * 12,
    0
)

# -----------------------------
# Customer Lifecycle Stage
# Useful for dashboard segmentation
# -----------------------------
prediction_table['LifecycleStage'] = pd.cut(
    prediction_table['TenureMonths'],
    bins=[0, 12, 24, 48, np.inf],
    labels=[
        'New',
        'Growing',
        'Established',
        'Loyal'
    ]
)

# -----------------------------
# Recommended Action
# -----------------------------
prediction_table['RecommendedAction'] = np.select(
    [
        prediction_table['RetentionPriority'] == 'Immediate Action',
        prediction_table['RetentionPriority'] == 'Monitor Closely',
        prediction_table['RetentionPriority'] == 'Nurture'
    ],
    [
        'Assign Customer Success Manager',
        'Launch Retention Campaign',
        'Increase Feature Adoption'
    ],
    default='Upsell Opportunity'
)

prediction_table.head()

,CustomerID,ActualChurn,ChurnProbability,MonthlyCharges,TenureMonths,RiskSegment,CustomerHealthScore,RetentionPriority,RevenueAtRisk,LifecycleStage,RecommendedAction
0,4376-KFVRS,0,0.184047,114.05,72,Low Risk,82,Healthy,0.0,Loyal,Upsell Opportunity
1,2754-SDJRD,0,0.803696,100.15,8,High Risk,20,Immediate Action,1201.8,New,Assign Customer Success Manager
2,9917-KWRBE,0,0.222799,78.35,41,Low Risk,78,Healthy,0.0,Established,Upsell Opportunity
3,0365-GXEZS,0,0.639094,78.20,18,High Risk,36,Monitor Closely,938.4,Growing,Launch Retention Campaign
4,9385-NXKDA,0,0.108792,82.65,72,Low Risk,89,Healthy,0.0,Loyal,Upsell Opportunity


In [17]:
customerpulse_summary = prediction_table.groupby(
    'RiskSegment',
    observed=False
).agg(
    Customers=('CustomerID', 'count'),
    AvgHealth=('CustomerHealthScore', 'mean'),
    RevenueAtRisk=('RevenueAtRisk', 'sum')
).reset_index()

customerpulse_summary

customerpulse_summary.to_csv(
    '../output_data/customerpulse_summary.csv',
    index=False
)

executive_kpis = pd.DataFrame({
    'Metric': [
        'Total Customers',
        'High Risk Customers',
        'Revenue At Risk',
        'Average Health Score'
    ],
    'Value': [
        len(prediction_table),
        (prediction_table['RiskSegment'] == 'High Risk').sum(),
        prediction_table['RevenueAtRisk'].sum(),
        round(
            prediction_table['CustomerHealthScore'].mean(),
            2
        )
    ]
})

executive_kpis

executive_kpis.to_csv(
    '../output_data/executive_kpis.csv',
    index=False
)

metrics_df.to_csv('../output_data/model_metrics.csv', index=False)
feature_importance.to_csv('../output_data/feature_importance.csv', index=False)
prediction_table.to_csv('../output_data/prediction_table.csv', index=False)

print('Exports completed.')


Exports completed.
